In [1]:
# ============================================================================
# 01 — FEASIBILITY: is the within-protein design possible?
# ----------------------------------------------------------------------------
# The strongest version of this study asks about the SAME protein twice: once
# where it does its usual thing, once where it does the opposite. The protein
# is then its own control — same name, same fame, same training frequency.
# This only works if enough proteins have BOTH signs. That is what we count.
# ============================================================================
import os, json, hashlib
from pathlib import Path
import pandas as pd, numpy as np
from dotenv import load_dotenv

load_dotenv()                      # reads .env; keys never appear in the code
ROOT = Path.cwd().parent           # notebooks/ -> project root
RAW  = ROOT/"data"/"raw"; RAW.mkdir(parents=True, exist_ok=True)

import omnipath as op

# ---- download and freeze -------------------------------------------------
COMPONENTS = {
    "causal_ppi":  lambda: op.interactions.OmniPath.get(organism="human", genesymbols=True),
    "collectri":   lambda: op.interactions.CollecTRI.get(organism="human", genesymbols=True),
    "kinaseextra": lambda: op.interactions.KinaseExtra.get(organism="human", genesymbols=True),
}
raw = {}
for name, fn in COMPONENTS.items():
    p = RAW/f"{name}.parquet"
    if p.exists():
        raw[name] = pd.read_parquet(p); tag = "cached"
    else:
        raw[name] = fn(); raw[name].to_parquet(p, index=False); tag = "downloaded"
    print(f"  {name:<14}{len(raw[name]):>8,} rows  [{tag}]")

# ---- filter: directed, signed one way, single proteins -------------------
def is_complex(s):
    return (not isinstance(s,str)) or (not s.strip()) or "COMPLEX" in s.upper() or "_" in s

NEEDS_DIR = {"causal_ppi": True, "kinaseextra": True, "collectri": False}
parts = []
for name, d in raw.items():
    d = d.copy()
    if NEEDS_DIR[name]:
        d = d[d.is_directed.astype(bool)]
    stim, inh = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    d = d[(stim ^ inh).values]                       # exactly one sign
    d = d[~d.source_genesymbol.map(is_complex).values]
    d = d[~d.target_genesymbol.map(is_complex).values]
    d = d[~d.source.astype(str).str.upper().str.contains("COMPLEX").values]
    d = d[~d.target.astype(str).str.upper().str.contains("COMPLEX").values].copy()
    d["sign"] = d.is_stimulation.astype(bool).map({True:"stimulates", False:"inhibits"})
    d["n_refs"] = (d.n_references.fillna(0).astype(int) if "n_references" in d.columns
                   else 0)
    d["component"] = name
    parts.append(d[["source_genesymbol","target_genesymbol","sign","n_refs","component"]])

E = pd.concat(parts, ignore_index=True)
print(f"\nsigned edges before dedup : {len(E):,}")

# remove pairs different sources sign both ways, then de-duplicate
conf = E.groupby(["source_genesymbol","target_genesymbol"]).sign.nunique()
bad  = set(conf[conf>1].index)
E["_pk"] = list(zip(E.source_genesymbol, E.target_genesymbol))
E = E[~E._pk.isin(bad)]
E = E.sort_values("n_refs", ascending=False).drop_duplicates("_pk", keep="first")
print(f"sign-conflicted pairs removed: {len(bad):,}")
print(f"usable signed edges          : {len(E):,}")

# ---- THE KEY COUNT -------------------------------------------------------
print("\n" + "="*74)
print("HOW MANY PROTEINS HAVE BOTH ACTIVATING AND INHIBITING TARGETS?")
print("="*74)

def summarise(min_refs, min_per_sign):
    e = E[E.n_refs >= min_refs]
    g = e.groupby(["source_genesymbol","sign"]).size().unstack(fill_value=0)
    for c in ("stimulates","inhibits"):
        if c not in g.columns: g[c] = 0
    ok = g[(g.stimulates >= min_per_sign) & (g.inhibits >= min_per_sign)]
    return len(ok), int(ok.min(axis=1).sum())

print(f"{'min refs/edge':>14}{'min per sign':>14}{'proteins':>11}{'matched pairs':>15}")
print("-"*54)
for mr in (1, 2, 3, 5):
    for mp in (1, 2, 3):
        n_prot, n_pairs = summarise(mr, mp)
        print(f"{mr:>14}{mp:>14}{n_prot:>11,}{n_pairs:>15,}")

print("\n  'matched pairs' = how many (usual, exception) question pairs we could")
print("  build. We need roughly 300. Read across to find a workable setting.")

# ---- how lopsided is each protein? ---------------------------------------
g = E.groupby(["source_genesymbol","sign"]).size().unstack(fill_value=0)
for c in ("stimulates","inhibits"):
    if c not in g.columns: g[c] = 0
g["total"] = g.stimulates + g.inhibits
g["stim_share"] = g.stimulates / g.total
big = g[g.total >= 5]
print("\n" + "="*74)
print("HOW LOPSIDED ARE PROTEINS?  (proteins with >=5 signed targets)")
print("="*74)
print(f"  proteins: {len(big):,}")
print(f"  >=80% one sign : {int(((big.stim_share>=.8)|(big.stim_share<=.2)).sum()):,} "
      f"({((big.stim_share>=.8)|(big.stim_share<=.2)).mean():.0%})")
print(f"  >=90% one sign : {int(((big.stim_share>=.9)|(big.stim_share<=.1)).sum()):,} "
      f"({((big.stim_share>=.9)|(big.stim_share<=.1)).mean():.0%})")
print("\n  Highly lopsided proteins are where the effect should be strongest:")
print("  a strong default, and a rare exception to it.")

print("\n" + "="*74)
print("EXAMPLE PROTEINS FOR THE WITHIN-PROTEIN DESIGN")
print("="*74)
cand = g[(g.total>=6) & (g.stimulates>=1) & (g.inhibits>=1) &
         ((g.stim_share>=.75)|(g.stim_share<=.25))].sort_values("total", ascending=False)
print(f"  candidates: {len(cand):,}\n")
print(f"{'protein':<12}{'stim':>6}{'inhib':>7}{'% stim':>8}   usual -> exception example")
for gene in cand.head(12).index:
    r = cand.loc[gene]
    usual = "stimulates" if r.stim_share>.5 else "inhibits"
    exc   = "inhibits" if usual=="stimulates" else "stimulates"
    ex = E[(E.source_genesymbol==gene) & (E.sign==exc)].sort_values("n_refs", ascending=False)
    t = ex.iloc[0].target_genesymbol if len(ex) else "?"
    print(f"{gene:<12}{int(r.stimulates):>6}{int(r.inhibits):>7}{r.stim_share:>8.0%}"
          f"   usually {usual}, but {exc} {t}")

c:\Users\tauhe\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  causal_ppi      86,189 rows  [cached]
  collectri       64,516 rows  [cached]
  kinaseextra     25,643 rows  [cached]

signed edges before dedup : 70,227
sign-conflicted pairs removed: 111
usable signed edges          : 61,975

HOW MANY PROTEINS HAVE BOTH ACTIVATING AND INHIBITING TARGETS?
 min refs/edge  min per sign   proteins  matched pairs
------------------------------------------------------
             1             1      1,511          8,577
             1             2        891          7,957
             1             3        630          7,435
             2             1        750          2,927
             2             2        388          2,565
             2             3        266          2,321
             3             1        505          1,642
             3             2        248          1,385
             3             3        153          1,195
             5             1        288            766
             5             2        129        

In [ ]:
# ============================================================================
# 01b — CAN WE BALANCE ACROSS PROTEIN TYPES AND DATA SOURCES?
# ----------------------------------------------------------------------------
# The example proteins from cell 01 were ALL transcription factors. If the whole
# benchmark is TFs, the finding is about gene regulation, not signalling.
# Here we count how many usable proteins are kinases, TFs, or other, and how
# much each OmniPath component contributes.
# ============================================================================
import urllib.request, urllib.parse, time, json
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
META = ROOT/"data"/"processed"; META.mkdir(parents=True, exist_ok=True)

# ---- which proteins are eligible at each strictness setting ---------------
def eligible(min_refs, min_per_sign):
    e = E[E.n_refs >= min_refs]
    g = e.groupby(["source_genesymbol","sign"]).size().unstack(fill_value=0)
    for c in ("stimulates","inhibits"):
        if c not in g.columns: g[c] = 0
    g["total"] = g.stimulates + g.inhibits
    g["stim_share"] = g.stimulates / g.total
    return g[(g.stimulates >= min_per_sign) & (g.inhibits >= min_per_sign)]

# working setting: >=2 refs per edge, >=2 targets of each sign
G = eligible(2, 2)
print(f"eligible proteins at (refs>=2, >=2 per sign): {len(G):,}")

# ---- where do their edges come from? -------------------------------------
sub = E[(E.n_refs >= 2) & E.source_genesymbol.isin(G.index)]
print("\nedges by component (for eligible proteins only):")
print("  " + sub.component.value_counts().to_string().replace("\n","\n  "))
print("\nproteins whose edges come ONLY from CollecTRI (pure TF):")
only_tf = sub.groupby("source_genesymbol").component.apply(
    lambda s: set(s) == {"collectri"})
print(f"  {int(only_tf.sum()):,} of {len(G):,} ({only_tf.mean():.0%})")

# ---- label each protein by function using UniProt --------------------------
acc = {}
for name in ["causal_ppi","collectri","kinaseextra"]:
    d = raw[name]
    for s, a in zip(d.source_genesymbol, d.source):
        if isinstance(a, str) and "COMPLEX" not in a.upper():
            acc.setdefault(s, a)

need = [acc[g] for g in G.index if g in acc]
print(f"\nfetching UniProt annotation for {len(need):,} proteins "
      f"(~{len(need)/50*1.2/60:.0f} min)")

cache = META/"protein_class.json"
if cache.exists():
    UP = json.loads(cache.read_text()); print("  cached")
else:
    UP = {}
    for i in range(0, len(need), 50):
        q = "(" + " OR ".join(f"accession:{a}" for a in need[i:i+50]) + ")"
        try:
            u = ("https://rest.uniprot.org/uniprotkb/stream?format=tsv"
                 "&fields=accession,protein_name,keyword,ec"
                 f"&query={urllib.parse.quote(q)}")
            txt = urllib.request.urlopen(u, timeout=120).read().decode()
            hdr = txt.splitlines()[0].split("\t")
            for ln in txt.splitlines()[1:]:
                p = ln.split("\t")
                if len(p) >= 2: UP[p[0]] = dict(zip(hdr, p))
        except Exception as ex:
            print(f"  batch {i} failed: {type(ex).__name__}")
        time.sleep(0.2)
    cache.write_text(json.dumps(UP)); print(f"  fetched {len(UP):,}")

def classify(rec):
    if not rec: return "unknown"
    kw = str(rec.get("Keywords","")).lower()
    nm = str(rec.get("Protein names","")).lower()
    ec = str(rec.get("EC number","")); blob = kw + " | " + nm
    if "ubl conjugation pathway" in kw or "ubiquitin-protein ligase" in nm or ec.startswith("2.3.2."): return "E3 ligase"
    if "protein phosphatase" in blob or ec.startswith("3.1.3."): return "phosphatase"
    if "kinase" in blob or ec.startswith(("2.7.1","2.7.10","2.7.11","2.7.12")): return "kinase"
    if "protease" in blob or "peptidase" in blob or ec.startswith("3.4."): return "protease"
    if any(k in kw for k in ["activator","repressor","dna-binding","transcription"]): return "transcription factor"
    return "other"

G = G.copy()
G["pclass"] = [classify(UP.get(acc.get(g))) for g in G.index]

print("\n" + "="*72); print("ELIGIBLE PROTEINS BY FUNCTION"); print("="*72)
vc = G.pclass.value_counts()
for k, v in vc.items():
    print(f"  {k:<22}{v:>6,}  ({v/len(G):>5.1%})")

print("\n  how many matched pairs could each class supply?")
print(f"  {'class':<22}{'proteins':>10}{'pairs':>10}{'median % usual':>16}")
print("  " + "-"*58)
for k in vc.index:
    s = G[G.pclass==k]
    pairs = int(s[["stimulates","inhibits"]].min(axis=1).sum())
    lop = np.maximum(s.stim_share, 1-s.stim_share).median()
    print(f"  {k:<22}{len(s):>10,}{pairs:>10,}{lop:>15.0%}")

print("\n" + "="*72); print("IS A BALANCED DESIGN POSSIBLE?"); print("="*72)
TARGET = 300
usable = vc[vc >= 15].index.tolist()
print(f"  classes with >=15 eligible proteins: {usable}")
per = TARGET // max(len(usable), 1)
print(f"  ~{per} pairs per class would give {per*len(usable)} total")
short = [k for k in usable
         if int(G[G.pclass==k][["stimulates","inhibits"]].min(axis=1).sum()) < per]
print(f"  classes that could NOT supply {per} pairs: {short if short else 'none'}")

print("\n  non-TF examples (these are what the benchmark needs):")
nt = G[(G.pclass.isin(["kinase","phosphatase","E3 ligase","protease"])) &
       (G.total>=6)].sort_values("total", ascending=False)
print(f"\n  {'protein':<12}{'class':<20}{'stim':>6}{'inhib':>7}{'% usual':>9}")
for gene in nt.head(15).index:
    r = nt.loc[gene]
    print(f"  {gene:<12}{r.pclass:<20}{int(r.stimulates):>6}{int(r.inhibits):>7}"
          f"{max(r.stim_share,1-r.stim_share):>9.0%}")

G.to_csv(META/"eligible_proteins.csv")
print(f"\nwritten: {META/'eligible_proteins.csv'}")

eligible proteins at (refs>=2, >=2 per sign): 388

edges by component (for eligible proteins only):
  component
  collectri      6994
  causal_ppi     3065
  kinaseextra    1815

proteins whose edges come ONLY from CollecTRI (pure TF):
  96 of 388 (25%)

fetching UniProt annotation for 388 proteins (~0 min)
  fetched 388

ELIGIBLE PROTEINS BY FUNCTION
  transcription factor     187  (48.2%)
  kinase                   122  (31.4%)
  other                     31  ( 8.0%)
  E3 ligase                 21  ( 5.4%)
  phosphatase               20  ( 5.2%)
  protease                   7  ( 1.8%)

  how many matched pairs could each class supply?
  class                   proteins     pairs  median % usual
  ----------------------------------------------------------
  transcription factor         187     1,236            76%
  kinase                       122     1,067            67%
  other                         31       100            67%
  E3 ligase                     21        61         

In [ ]:
# ============================================================================
# 01c — HOW STRICT CAN WE BE ABOUT "USUAL"?
# ----------------------------------------------------------------------------
# A protein that activates 58% of the time doesn't really have a habit. For the
# test to mean anything, the protein's default must be strong. Here we see how
# many proteins survive if we demand 70%, 80% or 90% one-directional, and
# whether every protein class can still fill its quota.
# ============================================================================
import pandas as pd, numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
G = pd.read_csv(ROOT/"data"/"processed"/"eligible_proteins.csv", index_col=0)
G["lopsided"] = np.maximum(G.stim_share, 1 - G.stim_share)
G["usual"]    = np.where(G.stim_share > 0.5, "stimulates", "inhibits")

print("="*78); print("EFFECT OF REQUIRING A STRONGER DEFAULT"); print("="*78)
print(f"{'min % usual':>12}{'proteins':>10}{'pairs':>8}   proteins per class")
print("-"*78)
for thr in (0.50, 0.60, 0.70, 0.75, 0.80, 0.90):
    s = G[G.lopsided >= thr]
    pairs = int(s[["stimulates","inhibits"]].min(axis=1).sum())
    by = s.pclass.value_counts()
    txt = "  ".join(f"{k[:12]}={v}" for k, v in by.items())
    print(f"{thr:>11.0%}{len(s):>10}{pairs:>8}   {txt}")

print("\n  We need ~60 pairs per class from >=15 proteins per class.")
print("  Pick the strictest threshold that still satisfies that.")

# ---- which way do the defaults point? ------------------------------------
print("\n" + "="*78)
print("DOES EVERY CLASS DEFAULT TO 'ACTIVATES'?")
print("="*78)
print("If some classes usually inhibit, the design gets stronger: the shortcut")
print("then points in different directions for different proteins, and a model")
print("that just always says 'activates' cannot mimic it.\n")
s = G[G.lopsided >= 0.70]
ct = pd.crosstab(s.pclass, s.usual)
for c in ("stimulates","inhibits"):
    if c not in ct.columns: ct[c] = 0
ct["% usually inhibits"] = (ct.inhibits/(ct.inhibits+ct.stimulates)*100).round(0)
print(ct.to_string())

print("\n  proteins whose DEFAULT is to inhibit (these are the valuable ones):")
inh = s[s.usual=="inhibits"].sort_values("total", ascending=False)
print(f"  {len(inh)} proteins\n")
print(f"  {'protein':<12}{'class':<22}{'stim':>6}{'inhib':>7}{'% usual':>9}")
for g in inh.head(15).index:
    r = inh.loc[g]
    print(f"  {g:<12}{r.pclass:<22}{int(r.stimulates):>6}{int(r.inhibits):>7}"
          f"{r.lopsided:>9.0%}")

# ---- the four question types the design produces -------------------------
print("\n" + "="*78)
print("THE FOUR CELLS OF THE DESIGN")
print("="*78)
print("Every question is one of these. Cells 2 and 3 are where models should")
print("fail if they are running on habit rather than knowledge.\n")
print(f"  {'#':<3}{'protein usually':<18}{'true answer here':<18}{'expected'}")
print("  " + "-"*64)
for i,(u,a,e) in enumerate([
    ("stimulates","stimulates","easy — habit agrees"),
    ("stimulates","inhibits",  "HARD — habit misleads"),
    ("inhibits",  "inhibits",  "easy — habit agrees"),
    ("inhibits",  "stimulates","HARD — habit misleads")], 1):
    print(f"  {i:<3}{u:<18}{a:<18}{e}")
print("\n  Balanced by design: equal numbers in all four cells, so a model that")
print("  always answers 'activates' scores exactly 50%, and so does a model")
print("  that perfectly follows the habit rule.")

n70 = G[G.lopsided>=0.70]
print(f"\n  at >=70% usual: {len(n70)} proteins, "
      f"{int(n70[['stimulates','inhibits']].min(axis=1).sum())} possible pairs")

EFFECT OF REQUIRING A STRONGER DEFAULT
 min % usual  proteins   pairs   proteins per class
------------------------------------------------------------------------------
        50%       388    2565   transcriptio=187  kinase=122  other=31  E3 ligase=21  phosphatase=20  protease=7
        60%       315    2025   transcriptio=167  kinase=90  other=24  E3 ligase=14  phosphatase=14  protease=6
        70%       201    1214   transcriptio=114  kinase=52  other=15  E3 ligase=12  phosphatase=7  protease=1
        75%       159     878   transcriptio=99  kinase=39  E3 ligase=8  other=7  phosphatase=5  protease=1
        80%       119     675   transcriptio=83  kinase=26  other=4  phosphatase=4  E3 ligase=2
        90%        33     181   transcriptio=23  kinase=6  phosphatase=2  other=1  E3 ligase=1

  We need ~60 pairs per class from >=15 proteins per class.
  Pick the strictest threshold that still satisfies that.

DOES EVERY CLASS DEFAULT TO 'ACTIVATES'?
If some classes usually inhibit, t

DIAGNOSTICS

In [2]:
# ============================================================================
# VERIFY — what did we actually download, and do the references hold up?
# ============================================================================
import pandas as pd, numpy as np, json
from pathlib import Path
ROOT = Path.cwd().parent; RAW = ROOT/"data"/"raw"; PROC = ROOT/"data"/"processed"

print("="*88); print("1. WHAT EACH COMPONENT ACTUALLY CONTAINS"); print("="*88)
for n in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{n}.parquet")
    refs = pd.to_numeric(d.get("n_references"), errors="coerce")
    print(f"\n{n}  ({len(d):,} rows)")
    print(f"  columns: {list(d.columns)[:12]}")
    if refs is not None:
        print(f"  has n_references : {refs.notna().mean():.1%} of rows")
        print(f"  n_references = 0 : {(refs.fillna(0)==0).mean():.1%}")
        print(f"  n_references >=2 : {(refs.fillna(0)>=2).mean():.1%}")
        print(f"  median | max     : {refs.median():.0f} | {refs.max():.0f}")
    if "sources" in d.columns:
        s = pd.Series([x for v in d.sources.dropna() for x in str(v).split(";")])
        print(f"  top resources    : {dict(s.value_counts().head(5))}")

print("\n"+"="*88); print("2. THE 37 KINASEEXTRA ITEMS IN THE BENCHMARK"); print("="*88)
B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)
k = B[B.component=="kinaseextra"]
print(f"  items: {len(k)}   reference counts: "
      f"min {k.n_refs.min()}, median {k.n_refs.median():.0f}, max {k.n_refs.max()}")
print(f"  all have >=2 references: {(k.n_refs>=2).all()}")
print("\n  by component, references in the final benchmark:")
print("  " + B.groupby("component").n_refs.agg(["count","min","median","max"])
      .to_string().replace("\n","\n  "))

print("\n"+"="*88); print("3. SNAPSHOT PROVENANCE — needed for Methods"); print("="*88)
m = RAW/"snapshot_manifest.json"
if m.exists():
    print(json.dumps(json.loads(m.read_text()), indent=2)[:900])
else:
    print("  no manifest found — record the download date manually")
    for n in ["causal_ppi","collectri","kinaseextra","all_interactions"]:
        p = RAW/f"{n}.parquet"
        if p.exists():
            import datetime
            print(f"    {n:<18}{len(pd.read_parquet(p)):>8,} rows   downloaded "
                  f"{datetime.datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d}")
try:
    import omnipath
    print(f"\n  omnipath python package version: {omnipath.__version__}")
except Exception as e:
    print(f"\n  version lookup failed: {e}")

1. WHAT EACH COMPONENT ACTUALLY CONTAINS

causal_ppi  (86,189 rows)
  columns: ['source', 'target', 'source_genesymbol', 'target_genesymbol', 'is_directed', 'is_stimulation', 'is_inhibition', 'consensus_direction', 'consensus_stimulation', 'consensus_inhibition', 'sources', 'references']
  has n_references : 100.0% of rows
  n_references = 0 : 0.0%
  n_references >=2 : 18.8%
  median | max     : 1 | 558
  top resources    : {'SIGNOR': np.int64(65532), 'HPRD': np.int64(8276), 'ProtMapper': np.int64(8274), 'Wang': np.int64(7916), 'SPIKE_LC': np.int64(7163)}

collectri  (64,516 rows)
  columns: ['source', 'target', 'source_genesymbol', 'target_genesymbol', 'is_directed', 'is_stimulation', 'is_inhibition', 'consensus_direction', 'consensus_stimulation', 'consensus_inhibition', 'sources', 'references']
  has n_references : 100.0% of rows
  n_references = 0 : 0.0%
  n_references >=2 : 31.0%
  median | max     : 1 | 380
  top resources    : {'CollecTRI': np.int64(64516), 'ExTRI_CollecTRI': np

In [4]:
# ============================================================================
# LOCK-IN — verify every claim in the "Sourcing interactions" paragraph
# ============================================================================
import json, datetime
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent; RAW = ROOT/"data"/"raw"; PROC = ROOT/"data"/"processed"
COMPS = {"causal_ppi":"core omnipath dataset",
         "collectri":"CollecTRI",
         "kinaseextra":"KinaseExtra"}

print("="*92); print("1. WHAT WAS DOWNLOADED, AND WHEN"); print("="*92)
try:
    import omnipath
    print(f"  omnipath package version : {omnipath.__version__}")
except Exception as e:
    print(f"  package version unavailable: {e}")
for n in list(COMPS)+["all_interactions"]:
    p = RAW/f"{n}.parquet"
    if not p.exists(): print(f"  {n:<18} MISSING"); continue
    d = pd.read_parquet(p)
    print(f"  {n:<18}{len(d):>9,} rows   downloaded "
          f"{datetime.datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d}")

print("\n"+"="*92); print("2. DO THE STORED TABLES MATCH A FRESH QUERY?"); print("="*92)
print("  Confirms the query parameters recorded in Methods are the ones used.")
print("  Row counts may differ if OmniPath has updated since 22 Aug 2026 —")
print("  that is expected and is why the snapshot exists.\n")
try:
    import omnipath as op
    FRESH = {"causal_ppi": op.interactions.OmniPath,
             "collectri":  op.interactions.CollecTRI,
             "kinaseextra":op.interactions.KinaseExtra}
    for n, fn in FRESH.items():
        try:
            live = fn.get(organism="human", genesymbols=True)
            old  = pd.read_parquet(RAW/f"{n}.parquet")
            same_cols = list(live.columns) == list(old.columns)
            print(f"  {n:<14} stored {len(old):>7,} | live {len(live):>7,} | "
                  f"columns identical: {same_cols}")
        except Exception as e:
            print(f"  {n:<14} live query failed: {type(e).__name__}")
except Exception as e:
    print(f"  omnipath import failed: {e}")

print("\n"+"="*92); print("3. HOW MANY PRIMARY RESOURCES FEED EACH COMPONENT?")
print("="*92)
print("  Counted from the 'sources' column of your own download. Use these")
print("  numbers rather than a figure quoted from the website.\n")
allres = set()
for n, lab in COMPS.items():
    d = pd.read_parquet(RAW/f"{n}.parquet")
    r = set()
    for v in d.sources.dropna():
        r.update(x.strip() for x in str(v).split(";") if x.strip())
    allres |= r
    print(f"  {lab:<28}{len(r):>4} distinct resources")
print(f"  {'union of the three':<28}{len(allres):>4}")
dall = RAW/"all_interactions.parquet"
if dall.exists():
    d = pd.read_parquet(dall); r = set()
    for v in d.sources.dropna():
        r.update(x.strip() for x in str(v).split(";") if x.strip())
    print(f"  {'complete interaction table':<28}{len(r):>4} resources, "
          f"{len(d):,} interactions")

print("\n"+"="*92); print("4. IS THE CORE DATASET SIGNED AND DIRECTED BY DEFINITION?")
print("="*92)
for n, lab in COMPS.items():
    d = pd.read_parquet(RAW/f"{n}.parquet")
    dirn = d.is_directed.astype(bool).mean() if "is_directed" in d else np.nan
    st, ih = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    print(f"  {lab:<28}directed {dirn:>6.1%} | signed one way {(st^ih).mean():>6.1%} "
          f"| unsigned {(~st & ~ih).mean():>6.1%} | both {(st & ih).mean():>6.1%}")
print("\n  'signed one way' is what the filter kept.")

print("\n"+"="*92); print("5. THE REFERENCE FILTER, COMPONENT BY COMPONENT"); print("="*92)
print(f"  {'component':<28}{'rows':>9}{'has ref':>9}{'>=2 refs':>10}{'median':>8}")
for n, lab in COMPS.items():
    d = pd.read_parquet(RAW/f"{n}.parquet")
    r = pd.to_numeric(d.n_references, errors="coerce").fillna(0)
    print(f"  {lab:<28}{len(d):>9,}{(r>0).mean():>9.1%}{(r>=2).mean():>10.1%}"
          f"{r.median():>8.0f}")
print("\n  The >=2 filter was applied identically to all three components.")

print("\n"+"="*92); print("6. THE FINAL BENCHMARK, BY COMPONENT"); print("="*92)
B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)
t = B.groupby("component").agg(items=("item_id","size"),
        min_refs=("n_refs","min"), median_refs=("n_refs","median"),
        max_refs=("n_refs","max"), proteins=("source","nunique"))
print(t.to_string())
print(f"\n  total items {len(B)} | unique source proteins {B.source.nunique()}")
print(f"  every item has >=2 references: {(B.n_refs>=2).all()}")

print("\n"+"="*92); print("7. NUMBERS YOU CAN WRITE, AND WHAT STILL NEEDS A SOURCE")
print("="*92)
print("  SAFE — measured from your own snapshot:")
for n, lab in COMPS.items():
    print(f"    {lab}: {len(pd.read_parquet(RAW/f'{n}.parquet')):,} interactions")
if dall.exists():
    print(f"    complete interaction table screened against: "
          f"{len(pd.read_parquet(dall)):,} interactions")
print(f"    resources feeding the three components used: {len(allres)}")
print("\n  STILL NEEDS A CITED SOURCE — do not write from memory:")
print("    - total OmniPath size across all five data domains")
print("    - the '168 resources' figure")
print("    - the OmniPath reference itself (Türei et al.)")
print("    Check https://omnipathdb.org and the OmniPath paper, and cite the")
print("    version matching a 22 Aug 2026 download.")

print("\n"+"="*92); print("8. WRITE THE MANIFEST SO THIS IS RECORDED"); print("="*92)
man = dict(downloaded="2026-08-22",
           omnipath_package=omnipath.__version__ if "omnipath" in dir() else "1.0.12",
           query_parameters=dict(organism="human", genesymbols=True),
           components={lab: int(len(pd.read_parquet(RAW/f"{n}.parquet")))
                       for n, lab in COMPS.items()},
           n_resources_union=len(allres),
           filters=["directed", "signed exactly one way",
                    "both partners single proteins",
                    ">=2 supporting publications",
                    "sign-conflicted pairs removed"])
(RAW/"snapshot_manifest.json").write_text(json.dumps(man, indent=2))
print(json.dumps(man, indent=2))
print(f"\n  written to {RAW/'snapshot_manifest.json'} — cite this in Methods")

1. WHAT WAS DOWNLOADED, AND WHEN
  omnipath package version : 1.0.12
  causal_ppi           86,189 rows   downloaded 2026-08-22
  collectri            64,516 rows   downloaded 2026-08-22
  kinaseextra          25,643 rows   downloaded 2026-08-22
  all_interactions    290,512 rows   downloaded 2026-08-22

2. DO THE STORED TABLES MATCH A FRESH QUERY?
  Confirms the query parameters recorded in Methods are the ones used.
  Row counts may differ if OmniPath has updated since 22 Aug 2026 —
  that is expected and is why the snapshot exists.

  causal_ppi     stored  86,189 | live  86,189 | columns identical: True
  collectri      stored  64,516 | live  64,516 | columns identical: True
  kinaseextra    stored  25,643 | live  25,643 | columns identical: True

3. HOW MANY PRIMARY RESOURCES FEED EACH COMPONENT?
  Counted from the 'sources' column of your own download. Use these
  numbers rather than a figure quoted from the website.

  core omnipath dataset        101 distinct resources
  Collec